# Label Error Detection — Label-Error Ground Truth

Defines label errors per dataset by comparing two annotation sets of the same images:

- **soft-GT (hard-thresholded)**: soft-label benchmark boxes whose maximum
  non-background class probability is > 0.5;
- **original hard annotations**: the dataset's official annotations in COCO format.

Both are matched with global greedy best-IoU-first, one-to-one, category-agnostic
matching at IoU ≥ 0.5. Every label error is a soft-GT box, of one of three types:

- **missing**: unmatched, max IoU with any original box ≤ 0.1;
- **misaligned**: unmatched, 0.1 < max IoU < 0.5 (unmatched boxes with max IoU ≥ 0.5,
  i.e. ones that lost a matching collision, are not counted);
- **classification**: matched with IoU ≥ 0.7, but with a different class than the
  original box.

The matching and error types follow the reference implementation in
`Uncertainty-aware-Object-Detection` (`helpers_legt.py::extract_label_errors`); see
`find_label_errors_reference_consistent` in `label_error_utils.py`.

In [1]:
import os
import sys
import json
import time
from collections import Counter

import numpy as np
import pandas as pd

from label_error_utils import (original_gt_to_hard_boxes, soft_gt_to_hard_boxes,
                                find_label_errors_reference_consistent, build_catid_to_idx)

os.makedirs("label_errors", exist_ok=True)

SOFT_LABEL_THRESH = 0.5
MATCH_IOU_THRESH = 0.5
CLASSIFICATION_IOU_THRESH = 0.7

# original_gt: the datasets' official hard annotations in COCO format. For VOC/Cityscapes/KITTI
# these are instances_*.json from inference/create_coco_style_annotations.ipynb, not the
# soft_instances_*.json files holding the soft-label benchmark's boxes.
DATASETS = {
    "coco_val": dict(
        display_name="COCO",
        original_gt="/home/klees/SoftTeacher/data/coco/annotations/instances_val2017.json",
        soft_gt="../data/soft_datasets/soft_COCO_2017_val.json",
    ),
    "cityscapes_trainval": dict(
        display_name="Cityscapes",
        original_gt="/home/klees/data/Cityscapes/trainval/instances_trainval.json",
        soft_gt="../data/soft_datasets/soft_Cityscapes_trainval_fixed.json",
    ),
    "kitti_train": dict(
        display_name="KITTI",
        original_gt="/home/klees/data/KITTI/train/instances_train.json",
        soft_gt="../data/soft_datasets/soft_Kitti_2D_train_fixed.json",
    ),
    "voc_trainval": dict(
        display_name="Pascal VOC",
        original_gt="/home/klees/data/VOC/VOCdevkit/VOC2012/instances_trainval.json",
        soft_gt="../data/soft_datasets/soft_PascalVOC_2012_detection_trainval_fixed.json",
    ),
}

/home/klees/miniconda3/envs/label_spreading/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
summary_rows = []
for dataset_key, cfg in DATASETS.items():
    t0 = time.time()
    with open(cfg["original_gt"]) as f:
        original_gt_json = json.load(f)
    original_by_image, filename_to_image_id = original_gt_to_hard_boxes(original_gt_json)
    soft_hard_by_image = soft_gt_to_hard_boxes(cfg["soft_gt"], filename_to_image_id,
                                                soft_label_thresh=SOFT_LABEL_THRESH)
    catid_to_idx = build_catid_to_idx(cfg["soft_gt"])

    label_errors = find_label_errors_reference_consistent(
        original_by_image, soft_hard_by_image, catid_to_idx,
        iou_threshold=MATCH_IOU_THRESH, classification_iou_threshold=CLASSIFICATION_IOU_THRESH)

    out_path = f"label_errors/{dataset_key}_label_errors.json"
    with open(out_path, "w") as f:
        json.dump(label_errors, f)

    type_counts = dict(Counter(e["type"] for e in label_errors))
    n_orig = sum(len(v) for v in original_by_image.values())
    n_soft_hard = sum(len(v) for v in soft_hard_by_image.values())
    summary_rows.append(dict(
        Dataset=cfg["display_name"], dataset_key=dataset_key,
        original_annotations=n_orig, soft_hard_annotations=n_soft_hard,
        label_errors=len(label_errors), **type_counts,
    ))
    print(f"{cfg['display_name']:12s}: original={n_orig:,}  soft-hard={n_soft_hard:,}  "
          f"label_errors={len(label_errors):,}  {type_counts}  [{time.time() - t0:.1f}s]  -> {out_path}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("label_errors/label_error_summary.csv", index=False)
summary_df

COCO        : original=36,781  soft-hard=52,542  label_errors=24,910  {'misaligned': 7959, 'missing': 16684, 'classification': 267}  [11.6s]  -> label_errors/coco_val_label_errors.json


Cityscapes  : original=62,649  soft-hard=65,849  label_errors=7,718  {'misaligned': 2384, 'missing': 5249, 'classification': 85}  [5.7s]  -> label_errors/cityscapes_trainval_label_errors.json


KITTI       : original=39,597  soft-hard=62,351  label_errors=29,966  {'missing': 21840, 'misaligned': 7119, 'classification': 1007}  [5.9s]  -> label_errors/kitti_train_label_errors.json


Pascal VOC  : original=31,561  soft-hard=35,240  label_errors=6,752  {'misaligned': 2293, 'missing': 4404, 'classification': 55}  [3.8s]  -> label_errors/voc_trainval_label_errors.json


,Dataset,dataset_key,original_annotations,soft_hard_annotations,label_errors,misaligned,missing,classification
0,COCO,coco_val,36781,52542,24910,7959,16684,267
1,Cityscapes,cityscapes_trainval,62649,65849,7718,2384,5249,85
2,KITTI,kitti_train,39597,62351,29966,7119,21840,1007
3,Pascal VOC,voc_trainval,31561,35240,6752,2293,4404,55
